# Mixed Similarity Feature Ablation

Compare probabilistic window features (P) against the same features plus two token-embedding similarity aggregates (P+S). The only feature difference is `mean_candidate_similarity__mean` and `mean_candidate_similarity__std`.

The expanded experiment trains LR and RF models for Gemma 270M, LittleLamb, Gemma 3 1B, and SmolLM2 360M, at matched 32/32 and 64/64 train/evaluation windows. Each scorer uses 50 texts per source/class stratum for training and another 50 for evaluation (300 documents per cohort). Gemma 270M and LittleLamb reuse the v2 cohort; Gemma 3 1B and SmolLM2 360M use separate cohorts because their available text IDs do not overlap the original sample. Within each scorer, P and P+S are evaluated on identical documents. Evaluation is only on the held-out Mixed Kaggle+MAGE+Outfox composition. Macro-F1 is reported over window predictions; bootstrap resampling samples documents within source/class strata and keeps each selected document's windows together.

Each fit uses a dedicated run ID. To repeat this experiment, set a new run ID before running cells that train or write artifacts.

In [ ]:
from datetime import datetime, timezone
from itertools import product
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

from notebooks.utils.classification_experiment_utils import discover_model_artifacts
from notebooks.utils.feature_aggregation_utils import (
    FeatureSet, ensure_prepared_partition, load_prepared_partition,
)
from notebooks.utils.model_training_utils import (
    fit_feature_vector_model, save_model_artifact, score_feature_vector_model,
)
from notebooks.utils.similarity_ablation_utils import (
    DOCUMENT_KEY_COLUMNS, FEATURE_SET_COLUMNS, MIXED_DATASETS, SPLIT_LABELS,
    intersect_eligible_texts_by_stratum, paired_macro_f1_bootstrap, sample_paired_text_cohorts,
)

In [ ]:
DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
PROCESSED_ROOT = DATA_ROOT / 'processed'
RUN_ID = 'mixed-similarity-ablation-2026-10-29-v6'
COHORT_SOURCE_RUN_ID = 'mixed-similarity-ablation-2026-10-29-v2'
SCORER_LLMS = ('gemma_3_270m_it', 'littlelamb', 'gemma_3_1b_it', 'smollm2_360m')
MODEL_FAMILIES = ('logistic_regression', 'random_forest')
WINDOWS = (32, 64)
STRIDE = 16
SAMPLE_SEED = 20261029
BOOTSTRAP_SEED = 20261030
BOOTSTRAP_REPLICATES = 1000
TEXTS_PER_DATASET_SPLIT_TRAIN = 50
TEXTS_PER_DATASET_SPLIT_EVALUATION = 50
FEATURE_SETS = {
    name: FeatureSet(
        f'similarity_ablation_{"P_plus_S" if name == "P+S" else "P"}',
        columns,
    )
    for name, columns in FEATURE_SET_COLUMNS.items()
}
MODEL_ROOT = DATA_ROOT / 'experiment-results' / 'classification' / 'models'
EVALUATION_ROOT = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / RUN_ID
REPORT_ROOT = DATA_ROOT / 'experiment-results' / 'report' / 'mixed-similarity-ablation' / RUN_ID
COHORT_SOURCE_REPORT_ROOT = DATA_ROOT / 'experiment-results' / 'report' / 'mixed-similarity-ablation' / COHORT_SOURCE_RUN_ID

if discover_model_artifacts(MODEL_ROOT, filters={'run_id': RUN_ID}):
    raise FileExistsError(f'Models already exist for {RUN_ID}; choose a new RUN_ID before training.')
if EVALUATION_ROOT.exists() or REPORT_ROOT.exists():
    raise FileExistsError(f'Evaluation/report outputs already exist for {RUN_ID}; choose a new RUN_ID.')

## Feature contract and paired text selection

The prepared feature cache is projected into two new, separately named feature sets. Eligible texts are intersected across windows and feature variants separately for each scorer. Gemma 270M and LittleLamb reuse the v2 cohort; Gemma 3 1B and SmolLM2 360M receive deterministic, disjoint cohorts of the same size because their available text IDs do not fully overlap the original pair. Within every scorer, P and P+S use identical train/evaluation texts.

In [ ]:
prepared_paths = {}
eligible_by_scorer_and_stratum = {scorer: {} for scorer in SCORER_LLMS}
for scorer_llm, dataset, split, window, feature_name in tqdm(
    product(SCORER_LLMS, MIXED_DATASETS, SPLIT_LABELS, WINDOWS, FEATURE_SETS),
    desc='Prepare ablation feature partitions',
):
    feature_set = FEATURE_SETS[feature_name]
    status = ensure_prepared_partition(
        DATA_ROOT, PROCESSED_ROOT, scorer_llm=scorer_llm, dataset=dataset, split=split,
        feature_set=feature_set, window_configuration=window, stride=STRIDE,
    )
    if status['status'] != 'available':
        raise FileNotFoundError(f'Cannot prepare {scorer_llm}/{dataset}/{split}/{window}/{feature_name}: {status.get("error")}')
    path = Path(status['path'])
    prepared_paths[(scorer_llm, dataset, split, window, feature_name)] = path
    available = pd.read_parquet(path / 'data.parquet', columns=['dataset', 'split', 'text_id', 'label'])
    eligible_by_scorer_and_stratum[scorer_llm].setdefault((dataset, split), []).append(available.drop_duplicates())

expected_strata = set(product(MIXED_DATASETS, SPLIT_LABELS))
eligible_frames_by_scorer = {}
for scorer_llm in SCORER_LLMS:
    eligible = intersect_eligible_texts_by_stratum(eligible_by_scorer_and_stratum[scorer_llm])
    actual_strata = set(eligible[['dataset', 'split']].itertuples(index=False, name=None))
    if actual_strata != expected_strata:
        raise ValueError(f'Incomplete Mixed strata for {scorer_llm}: expected {expected_strata}, found {actual_strata}')
    eligible_frames_by_scorer[scorer_llm] = eligible
eligible_texts = pd.concat([frame.assign(scorer_llm=scorer) for scorer, frame in eligible_frames_by_scorer.items()], ignore_index=True)

reference_texts = pd.read_csv(COHORT_SOURCE_REPORT_ROOT / 'selected_texts.csv')
selected_frames = []
for scorer_llm in SCORER_LLMS[:2]:
    selected = reference_texts.assign(scorer_llm=scorer_llm)
    eligible = eligible_frames_by_scorer[scorer_llm]
    requested_keys = set(map(tuple, selected[list(DOCUMENT_KEY_COLUMNS)].itertuples(index=False, name=None)))
    eligible_keys = set(map(tuple, eligible[list(DOCUMENT_KEY_COLUMNS)].itertuples(index=False, name=None)))
    if requested_keys.difference(eligible_keys):
        raise ValueError(f'The v2 text cohort is not fully available for {scorer_llm}; use a fresh cohort for all scorers.')
    selected_frames.append(selected)
for scorer_llm in SCORER_LLMS[2:]:
    selected = sample_paired_text_cohorts(
        eligible_frames_by_scorer[scorer_llm],
        train_texts_per_stratum=TEXTS_PER_DATASET_SPLIT_TRAIN,
        evaluation_texts_per_stratum=TEXTS_PER_DATASET_SPLIT_EVALUATION,
        seed=SAMPLE_SEED,
    )
    selected_frames.append(selected.assign(scorer_llm=scorer_llm))
selected_texts = pd.concat(selected_frames, ignore_index=True)
sample_counts = selected_texts.groupby(['scorer_llm', 'partition', 'dataset', 'split'], as_index=False).agg(text_count=('text_id', 'nunique'))
assert sample_counts['text_count'].eq(50).all()
assert selected_texts.groupby(['scorer_llm', 'partition']).size().eq(300).all()
display(sample_counts)
display(selected_texts.groupby('partition').size().rename('selected_text_count'))

## Train and evaluate the paired window classifiers

Each P/P+S model pair shares scorer, classifier family, training window, seed, and exact training/evaluation text IDs. LR and RF use the existing project training pipelines and fixed hyperparameters. Model artifacts are saved under this notebook's unique run ID.

In [ ]:
def load_selected_rows(path, selected, feature_columns):
    metadata = json.loads((path / 'metadata.json').read_text(encoding='utf-8'))
    frame = load_prepared_partition(path, expected=metadata)
    selected_keys = selected[[*DOCUMENT_KEY_COLUMNS, 'partition']].drop_duplicates()
    result = frame.merge(selected_keys, on=list(DOCUMENT_KEY_COLUMNS), how='inner', validate='many_to_one')
    if result.empty:
        raise ValueError(f'No selected rows found in prepared partition {path}')
    missing = set(feature_columns).difference(result.columns)
    if missing:
        raise ValueError(f'Prepared partition {path} is missing features: {sorted(missing)}')
    return result

prediction_frames = []
manifest_models = []
for scorer_llm, model_family, window in tqdm(
    product(SCORER_LLMS, MODEL_FAMILIES, WINDOWS), desc='Fit/evaluate model configurations'
):
    scorer_texts = selected_texts.loc[selected_texts['scorer_llm'].eq(scorer_llm)]
    train_texts = scorer_texts.loc[scorer_texts['partition'].eq('train')]
    evaluation_texts = scorer_texts.loc[scorer_texts['partition'].eq('evaluation')]
    for feature_name, feature_set in FEATURE_SETS.items():
        train_parts, evaluation_parts = [], []
        for dataset in MIXED_DATASETS:
            for split in SPLIT_LABELS:
                path = prepared_paths[(scorer_llm, dataset, split, window, feature_name)]
                train_selection = train_texts.loc[train_texts['dataset'].eq(dataset) & train_texts['split'].eq(split)]
                evaluation_selection = evaluation_texts.loc[evaluation_texts['dataset'].eq(dataset) & evaluation_texts['split'].eq(split)]
                if not train_selection.empty:
                    train_parts.append(load_selected_rows(path, train_selection, feature_set.source_columns))
                if not evaluation_selection.empty:
                    evaluation_parts.append(load_selected_rows(path, evaluation_selection, feature_set.source_columns))
        train_rows = pd.concat(train_parts, ignore_index=True)
        evaluation_rows = pd.concat(evaluation_parts, ignore_index=True)
        if train_rows.groupby(list(DOCUMENT_KEY_COLUMNS))['label'].nunique().gt(1).any():
            raise ValueError('Training documents contain conflicting labels')
        if evaluation_rows.groupby(list(DOCUMENT_KEY_COLUMNS))['label'].nunique().gt(1).any():
            raise ValueError('Evaluation documents contain conflicting labels')
        train_keys = set(map(tuple, train_rows[list(DOCUMENT_KEY_COLUMNS)].drop_duplicates().itertuples(index=False, name=None)))
        evaluation_keys = set(map(tuple, evaluation_rows[list(DOCUMENT_KEY_COLUMNS)].drop_duplicates().itertuples(index=False, name=None)))
        if train_keys & evaluation_keys:
            raise ValueError('Training/evaluation document overlap detected')
        fitted = fit_feature_vector_model(
            model_family, train_rows, feature_columns=feature_set.source_columns, seed=SAMPLE_SEED
        )
        train_parent_count = len(train_keys)
        metadata = {
            'artifact_version': 1, 'run_id': RUN_ID, 'created_at': datetime.now(timezone.utc).isoformat(),
            'model_family': model_family, 'scorer_llm': scorer_llm,
            'training_datasets': list(MIXED_DATASETS), 'training_regime': 'Kaggle+MAGE+Outfox',
            'training_composition': 'Kaggle+MAGE+Outfox', 'training_window': str(window),
            'feature_set': feature_set.name, 'feature_columns': list(feature_set.source_columns),
            'cohort_seed': SAMPLE_SEED, 'max_per_split': TEXTS_PER_DATASET_SPLIT_TRAIN,
            'min_text_tokens': None, 'stride': STRIDE, 'train_row_count': len(train_rows),
            'validation_row_count': 0, 'train_parent_count': train_parent_count,
            'validation_parent_count': 0, 'train_hg_row_count': int(train_rows['label'].eq(0).sum()),
            'train_mg_row_count': int(train_rows['label'].eq(1).sum()),
            'validation_hg_row_count': 0, 'validation_mg_row_count': 0,
            'evaluation_text_count': len(evaluation_keys),
            'similarity_ablation_feature_variant': feature_name,
        }
        save_model_artifact(
            DATA_ROOT, fitted, metadata=metadata, run_id=RUN_ID,
            training_metrics=pd.DataFrame(), training_coverage=pd.DataFrame(),
        )
        scored = score_feature_vector_model(fitted, evaluation_rows)
        scored = scored.assign(
            scorer_llm=scorer_llm, model_family=model_family, window=window,
            feature_set=feature_name, window_id=scored['start_token'],
        )
        prediction_frames.append(scored)
        manifest_models.append({
            'scorer_llm': scorer_llm, 'model_family': model_family, 'window': window,
            'feature_set': feature_name, 'feature_columns': list(feature_set.source_columns),
            'train_documents': train_parent_count, 'evaluation_documents': len(evaluation_keys),
            'train_windows': len(train_rows), 'evaluation_windows': len(evaluation_rows),
        })

window_predictions = pd.concat(prediction_frames, ignore_index=True)
window_predictions['prediction'] = window_predictions['prediction'].astype(int)
bootstrap_metrics = paired_macro_f1_bootstrap(
    window_predictions, n_bootstrap=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED
)
display(bootstrap_metrics)

## Paired macro-F1 comparison

The paired point plot shows the P and P+S macro-F1 for each window configuration. Labels report the paired P+S minus P difference and its 95% document-clustered bootstrap interval.

In [ ]:
figure, axes = plt.subplots(len(SCORER_LLMS), len(MODEL_FAMILIES), figsize=(12, 16), sharey=True, constrained_layout=True, squeeze=False)
colors = {32: 'tab:blue', 64: 'tab:orange'}
for row, scorer_llm in enumerate(SCORER_LLMS):
    for column, model_family in enumerate(MODEL_FAMILIES):
        axis = axes[row, column]
        selected = bootstrap_metrics.loc[
            bootstrap_metrics['scorer_llm'].eq(scorer_llm)
            & bootstrap_metrics['model_family'].eq(model_family)
        ]
        for result in selected.itertuples(index=False):
            color = colors[int(result.window)]
            axis.plot([0, 1], [result.macro_f1_P, result.macro_f1_P_plus_S], marker='o', color=color, label=f'window {result.window}')
            axis.annotate(
                f'Δ={result.delta_macro_f1:+.3f} [{result.ci_lower:+.3f}, {result.ci_upper:+.3f}]',
                (1, result.macro_f1_P_plus_S), xytext=(5, 0), textcoords='offset points',
                va='center', fontsize=8, color=color,
            )
        axis.set_xticks([0, 1], ['P', 'P+S'])
        axis.set_ylim(0, 1)
        axis.set_title(f'{scorer_llm} — {model_family.replace("_", " ")}')
        axis.set_ylabel('Window-level macro F1')
        axis.grid(axis='y', alpha=0.25)
        axis.legend(title='Matched window', frameon=False)
figure.suptitle('Mixed held-out set: isolated contribution of similarity features')
display(figure)
plt.close(figure)

In [ ]:
REPORT_ROOT.mkdir(parents=True, exist_ok=False)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=False)
window_predictions.sort_values(
    ['scorer_llm', 'model_family', 'window', 'feature_set', *DOCUMENT_KEY_COLUMNS, 'window_id'],
    kind='stable',
).to_parquet(EVALUATION_ROOT / 'mixed_window_predictions.parquet', index=False)
bootstrap_metrics.to_csv(REPORT_ROOT / 'paired_macro_f1_bootstrap.csv', index=False)
sample_counts.to_csv(REPORT_ROOT / 'sample_counts.csv', index=False)
selected_texts.to_csv(REPORT_ROOT / 'selected_texts.csv', index=False)
pd.DataFrame(manifest_models).to_csv(REPORT_ROOT / 'model_manifest.csv', index=False)
figure.savefig(REPORT_ROOT / 'paired_macro_f1.png', dpi=180, bbox_inches='tight')
figure.savefig(REPORT_ROOT / 'paired_macro_f1.svg', bbox_inches='tight')
manifest = {
    'run_id': RUN_ID, 'created_at': datetime.now(timezone.utc).isoformat(),
    'training_composition': 'Kaggle+MAGE+Outfox', 'evaluation_composition': 'Kaggle+MAGE+Outfox',
    'scorers': SCORER_LLMS, 'model_families': MODEL_FAMILIES,
    'matched_training_evaluation_windows': WINDOWS, 'stride': STRIDE,
    'feature_sets': {name: list(feature_set.source_columns) for name, feature_set in FEATURE_SETS.items()},
    'train_texts_per_dataset_split': TEXTS_PER_DATASET_SPLIT_TRAIN,
    'evaluation_texts_per_dataset_split': TEXTS_PER_DATASET_SPLIT_EVALUATION,
    'training_text_count': int(selected_texts['partition'].eq('train').sum()),
    'evaluation_text_count': int(selected_texts['partition'].eq('evaluation').sum()),
    'sampling_seed': SAMPLE_SEED, 'bootstrap_seed': BOOTSTRAP_SEED,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'bootstrap_unit': 'document, stratified by source dataset and HG/MG label; all windows retained together',
    'metric_unit': 'window', 'primary_metric': 'macro_f1',
    'classifier_hyperparameters': {
        'logistic_regression': {'class_weight': 'balanced', 'max_iter': 1000, 'C': '1.0 (scikit-learn default)', 'random_state': SAMPLE_SEED, 'preprocessing': ['median imputation', 'standard scaling']},
        'random_forest': {'n_estimators': 100, 'class_weight': 'balanced', 'random_state': SAMPLE_SEED, 'n_jobs': -1, 'preprocessing': ['median imputation']},
    },
    'models': manifest_models,
}
(REPORT_ROOT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str) + '\n', encoding='utf-8')
print(f'Models: {MODEL_ROOT}')
print(f'Evaluation predictions: {EVALUATION_ROOT}')
print(f'Report outputs: {REPORT_ROOT}')